In [ ]:
# ─── Standard Library ───────────────────────────────────────────────────────────
from pathlib import Path
from collections import Counter

# ─── Geospatial Processing ──────────────────────────────────────────────────────
import geopandas as gpd
import pandas as pd

# ─── Network Analysis ───────────────────────────────────────────────────────────
import networkx as nx
import pandana

# ─── Custom Modules ────────────────────────────────────────────────────────────
import geo_util as gu

In [ ]:
# global definitions
_data_path = Path( './data/' )

In [ ]:
nodes_df = gpd.read_parquet(_data_path.joinpath('pandana_nodes.parquet')).set_index('id').to_crs(gu.projectedCrsNL)
edges_df = pd.read_parquet(_data_path.joinpath('pandana_edges.parquet'))

In [ ]:
G = nx.from_pandas_edgelist(edges_df, source='u', target='v', create_using=nx.Graph)
cm = {
    node: idx
    for idx, component in enumerate(nx.connected_components(G))
    for node in component
}

In [ ]:
nodes_df['component'] = nodes_df.index.map(cm)

In [ ]:
nodes_filtered = nodes_df[ nodes_df.component == nodes_df.component.value_counts().idxmax() ]

In [ ]:
nodes_filtered.shape, nodes_df.shape

In [ ]:
edges_filtered = edges_df[
    edges_df['u'].isin(nodes_filtered.index) & edges_df['v'].isin(nodes_filtered.index)
]

In [ ]:
edges_filtered.shape, edges_df.shape

In [ ]:
net = pandana.Network(
    nodes_filtered.geometry.x,
    nodes_filtered.geometry.y,
    edges_filtered['u'],
    edges_filtered['v'],
    edges_filtered[['length']],
    twoway=True
)

In [ ]:
net.save_hdf5(_data_path.joinpath('nl_pandana_network.h5'))

In [ ]:
net.nodes_df.to_parquet(_data_path.joinpath('nl_pandana_nodes_df.parquet'))
net.edges_df.to_parquet(_data_path.joinpath('nl_pandana_edges_df.parquet'))